# Example 3.2 / 1: Gate result processing

Companion notebook for **3.2 Did It Solve? Solve Results** in the AMPL lab Documentation, section "Gate result processing". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Gate result processing

This example deliberately has no feasible solution. The status is checked before any value of
`x` is used.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var x;
minimize Cost: x;
subject to Lower: x >= 2;
subject to Upper: x <= 1;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
# This model has no external data.

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)
print(int(ampl.get_value("solve_result_num")))
if ampl.solve_result == "solved":
    print(ampl.var["x"].value())
else:
    print("No solution values used")

[WARNING] 
	presolve, variable x:
		impossible deduced bounds: lower = 2, upper = 1;
		difference = 1


infeasible
299
No solution values used


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
# This model has no external data.

Writing data.dat


In a Python session, amplpy reports AMPL's presolve warning as an `AMPLException` after the commands have run, and a warning left from the previous solve can surface at the next step. Each step below therefore catches the exception and prints it. At the AMPL command line the same warning is only printed, as on the page.

In [7]:
from amplpy import AMPLException

try:
    ampl.eval(r"""
reset;
model model.mod;
data data.dat;
""")
except AMPLException as warning:
    print("AMPL warning:", warning)

AMPL warning: presolve, variable x:
	impossible deduced bounds: lower = 2, upper = 1;
	difference = 1


In [8]:
# model.run
try:
    ampl.eval(r"""
option solver highs;
solve;
display solve_result, solve_result_num;
if solve_result = "solved" then display x;
else print "No solution values used";
""")
except AMPLException as warning:
    print("AMPL warning:", warning)

[WARNING] 
	presolve, variable x:
		impossible deduced bounds: lower = 2, upper = 1;
		difference = 1


solve_result = infeasible
solve_result_num = 299

No solution values used
AMPL warning: presolve, variable x:
	impossible deduced bounds: lower = 2, upper = 1;
	difference = 1
